In [4]:
import sqlite3
import hashlib
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

def sha256(val):
    return hashlib.sha256(str(val).encode()).hexdigest()

def execute_table_load(conn, update_sql, insert_sql, table_name):
    cursor = conn.cursor()
    try:
        cursor.execute(f'SELECT COUNT(*) FROM "{table_name}"')
        before_count = cursor.fetchone()[0]

        cursor.execute(update_sql)
        updated = cursor.rowcount

        cursor.execute(insert_sql)
        inserted = cursor.rowcount

        conn.commit()

        cursor.execute(f'SELECT COUNT(*) FROM "{table_name}"')
        after_count = cursor.fetchone()[0]

        if updated == 0 and inserted == 0:
            message = f"{table_name}: {before_count} rows already in DB | no changes detected, rows already up to date"
        else:
            message = f"{table_name}: {before_count} rows already in DB | {updated} rows closed off | {inserted} rows inserted | {after_count} total rows now in DB"

        print(message)
        logger.info(message)
        return updated, inserted, True

    except Exception as e:
        error_message = f"{table_name} FAILED and rolled back: {str(e)}"
        print(error_message)
        logger.error(error_message)
        conn.rollback()
        return 0, 0, False


def run_silver_loads(db_path):
    conn = sqlite3.connect(db_path)
    conn.create_function("sha256", 1, sha256)
    message = f"Connected to: {db_path}"
    print(message)
    logger.info(message)

    tables = [
        (
            "silver_games",
            """UPDATE "silver_games"
            SET ChangeEffectiveTo = DATETIME('now')
            WHERE Game_ID IN (
                SELECT sg.Game_ID
                FROM "silver_games" sg
                INNER JOIN "stg_games" stg ON stg.id = sg.Game_ID
                WHERE sg.ChangeEffectiveTo = '9999-12-31 23:59:59'
                AND sg.Hashdiff <> sha256(
                    COALESCE(stg.name,'') || '|' || COALESCE(stg.description,'') || '|' || 
                    COALESCE(stg.publisher,'') || '|' || COALESCE(stg.developer,'') || '|' || 
                    COALESCE(stg.released_date,'')
                )
            )""",
            """INSERT INTO "silver_games" (
                "Game_ID", "Game_Name", "Game_Description", "Publisher_Name", 
                "Developer_Company", "released_date",
                "ChangeEffectiveFrom", "ChangeEffectiveTo", "Hashdiff"
            )
            SELECT 
                stg.id, stg.name, stg.description, stg.publisher, stg.developer, stg.released_date,
                DATETIME('now'), '9999-12-31 23:59:59',
                sha256(
                    COALESCE(stg.name,'') || '|' || COALESCE(stg.description,'') || '|' || 
                    COALESCE(stg.publisher,'') || '|' || COALESCE(stg.developer,'') || '|' || 
                    COALESCE(stg.released_date,'')
                )
            FROM "stg_games" stg
            LEFT JOIN "silver_games" sg ON stg.id = sg.Game_ID AND sg.ChangeEffectiveTo = '9999-12-31 23:59:59'
            WHERE sg.Game_ID IS NULL
               OR sg.Hashdiff <> sha256(
                    COALESCE(stg.name,'') || '|' || COALESCE(stg.description,'') || '|' || 
                    COALESCE(stg.publisher,'') || '|' || COALESCE(stg.developer,'') || '|' || 
                    COALESCE(stg.released_date,'')
                )"""
        ),
        (
            "silver_character",
            """UPDATE "silver_character"
            SET ChangeEffectiveTo = DATETIME('now')
            WHERE Character_ID IN (
                SELECT sc.Character_ID
                FROM "silver_character" sc
                INNER JOIN "stg_characters" stg ON stg.id = sc.Character_ID
                WHERE sc.ChangeEffectiveTo = '9999-12-31 23:59:59'
                AND sc.HashDiff <> sha256(
                    COALESCE(stg.name,'') || '|' || COALESCE(stg.description,'') || '|' || 
                    COALESCE(stg.gender,'') || '|' || COALESCE(stg.race,'')
                )
            )""",
            """INSERT INTO "silver_character" (
                "Character_ID", "Character_Name", "Character_Description",
                "Character_Gender", "Character_Race",
                "ChangeEffectiveFrom", "ChangeEffectiveTo", "HashDiff"
            )
            SELECT 
                stg.id, stg.name, stg.description, stg.gender, stg.race,
                DATETIME('now'), '9999-12-31 23:59:59',
                sha256(
                    COALESCE(stg.name,'') || '|' || COALESCE(stg.description,'') || '|' || 
                    COALESCE(stg.gender,'') || '|' || COALESCE(stg.race,'')
                )
            FROM "stg_characters" stg
            LEFT JOIN "silver_character" sc ON stg.id = sc.Character_ID AND sc.ChangeEffectiveTo = '9999-12-31 23:59:59'
            WHERE sc.Character_ID IS NULL
               OR sc.HashDiff <> sha256(
                    COALESCE(stg.name,'') || '|' || COALESCE(stg.description,'') || '|' || 
                    COALESCE(stg.gender,'') || '|' || COALESCE(stg.race,'')
                )"""
        ),
        (
            "silver_dungeons",
            """UPDATE "silver_dungeons"
            SET ChangeEffectiveTo = DATETIME('now')
            WHERE Dungeon_ID IN (
                SELECT sd.Dungeon_ID
                FROM "silver_dungeons" sd
                INNER JOIN "stg_dungeons" stg ON stg.id = sd.Dungeon_ID
                WHERE sd.ChangeEffectiveTo = '9999-12-31 23:59:59'
                AND sd.HashDiff <> sha256(
                    COALESCE(stg.name,'') || '|' || COALESCE(stg.description,'')
                )
            )""",
            """INSERT INTO "silver_dungeons" (
                "Dungeon_ID", "Dungeon_Name", "Dungeon_Description",
                "ChangeEffectiveFrom", "ChangeEffectiveTo", "HashDiff"
            )
            SELECT 
                stg.id, stg.name, stg.description,
                DATETIME('now'), '9999-12-31 23:59:59',
                sha256(COALESCE(stg.name,'') || '|' || COALESCE(stg.description,''))
            FROM "stg_dungeons" stg
            LEFT JOIN "silver_dungeons" sd ON stg.id = sd.Dungeon_ID AND sd.ChangeEffectiveTo = '9999-12-31 23:59:59'
            WHERE sd.Dungeon_ID IS NULL
               OR sd.HashDiff <> sha256(COALESCE(stg.name,'') || '|' || COALESCE(stg.description,''))"""
        ),
        (
            "silver_Bosses",
            """UPDATE "silver_Bosses"
            SET ChangeEffectiveTo = DATETIME('now')
            WHERE Boss_ID IN (
                SELECT sb.Boss_ID
                FROM "silver_Bosses" sb
                INNER JOIN "stg_bosses" stg ON stg.id = sb.Boss_ID
                WHERE sb.ChangeEffectiveTo = '9999-12-31 23:59:59'
                AND sb.HashDiff <> sha256(
                    COALESCE(stg.name,'') || '|' || COALESCE(stg.description,'')
                )
            )""",
            """INSERT INTO "silver_Bosses" (
                "Boss_ID", "Boss_Name", "Boss_Description",
                "ChangeEffectiveFrom", "ChangeEffectiveTo", "HashDiff"
            )
            SELECT 
                stg.id, stg.name, stg.description,
                DATETIME('now'), '9999-12-31 23:59:59',
                sha256(COALESCE(stg.name,'') || '|' || COALESCE(stg.description,''))
            FROM "stg_bosses" stg
            LEFT JOIN "silver_Bosses" sb ON stg.id = sb.Boss_ID AND sb.ChangeEffectiveTo = '9999-12-31 23:59:59'
            WHERE sb.Boss_ID IS NULL
               OR sb.HashDiff <> sha256(COALESCE(stg.name,'') || '|' || COALESCE(stg.description,''))"""
        ),
        (
            "silver_monsters",
            """UPDATE "silver_monsters"
            SET ChangeEffectiveTo = DATETIME('now')
            WHERE Monster_ID IN (
                SELECT sm.Monster_ID
                FROM "silver_monsters" sm
                INNER JOIN "stg_monsters" stg ON stg.id = sm.Monster_ID
                WHERE sm.ChangeEffectiveTo = '9999-12-31 23:59:59'
                AND sm.Hashdiff <> sha256(
                    COALESCE(stg.name,'') || '|' || COALESCE(stg.description,'')
                )
            )""",
            """INSERT INTO "silver_monsters" (
                "Monster_ID", "Monster_Name", "Monster_Description",
                "ChangeEffectiveFrom", "ChangeEffectiveTo", "Hashdiff"
            )
            SELECT 
                stg.id, stg.name, stg.description,
                DATETIME('now'), '9999-12-31 23:59:59',
                sha256(COALESCE(stg.name,'') || '|' || COALESCE(stg.description,''))
            FROM "stg_monsters" stg
            LEFT JOIN "silver_monsters" sm ON stg.id = sm.Monster_ID AND sm.ChangeEffectiveTo = '9999-12-31 23:59:59'
            WHERE sm.Monster_ID IS NULL
               OR sm.Hashdiff <> sha256(COALESCE(stg.name,'') || '|' || COALESCE(stg.description,''))"""
        ),
        (
            "silver_items",
            """UPDATE "silver_items"
            SET ChangeEffectiveTo = DATETIME('now')
            WHERE Item_ID IN (
                SELECT si.Item_ID
                FROM "silver_items" si
                INNER JOIN "stg_items" stg ON stg.id = si.Item_ID
                WHERE si.ChangeEffectiveTo = '9999-12-31 23:59:59'
                AND si.Hashdiff <> sha256(
                    COALESCE(stg.name,'') || '|' || COALESCE(stg.description,'')
                )
            )""",
            """INSERT INTO "silver_items" (
                "Item_ID", "Item_Name", "Item_description",
                "ChangeEffectiveFrom", "ChangeEffectiveTo", "Hashdiff"
            )
            SELECT 
                stg.id, stg.name, stg.description,
                DATETIME('now'), '9999-12-31 23:59:59',
                sha256(COALESCE(stg.name,'') || '|' || COALESCE(stg.description,''))
            FROM "stg_items" stg
            LEFT JOIN "silver_items" si ON stg.id = si.Item_ID AND si.ChangeEffectiveTo = '9999-12-31 23:59:59'
            WHERE si.Item_ID IS NULL
               OR si.Hashdiff <> sha256(COALESCE(stg.name,'') || '|' || COALESCE(stg.description,''))"""
        ),
        (
            "silver_places",
            """UPDATE "silver_places"
            SET ChangeEffectiveTo = DATETIME('now')
            WHERE Place_ID IN (
                SELECT sp.Place_ID
                FROM "silver_places" sp
                INNER JOIN "stg_places" stg ON stg.id = sp.Place_ID
                WHERE sp.ChangeEffectiveTo = '9999-12-31 23:59:59'
                AND sp.Hashdiff <> sha256(
                    COALESCE(stg.name,'') || '|' || COALESCE(stg.description,'')
                )
            )""",
            """INSERT INTO "silver_places" (
                "Place_ID", "Place_Name", "Place_Description",
                "ChangeEffectiveFrom", "ChangeEffectiveTo", "Hashdiff"
            )
            SELECT 
                stg.id, stg.name, stg.description,
                DATETIME('now'), '9999-12-31 23:59:59',
                sha256(COALESCE(stg.name,'') || '|' || COALESCE(stg.description,''))
            FROM "stg_places" stg
            LEFT JOIN "silver_places" sp ON stg.id = sp.Place_ID AND sp.ChangeEffectiveTo = '9999-12-31 23:59:59'
            WHERE sp.Place_ID IS NULL
               OR sp.Hashdiff <> sha256(COALESCE(stg.name,'') || '|' || COALESCE(stg.description,''))"""
        ),
        (
            "silver_staff",
            """UPDATE "silver_staff"
            SET ChangeEffectiveTo = DATETIME('now')
            WHERE Staff_ID IN (
                SELECT ss.Staff_ID
                FROM "silver_staff" ss
                INNER JOIN "stg_staff" stg ON stg.id = ss.Staff_ID
                WHERE ss.ChangeEffectiveTo = '9999-12-31 23:59:59'
                AND ss.HashDiff <> sha256(COALESCE(stg.name,''))
            )""",
            """INSERT INTO "silver_staff" (
                "Staff_ID", "Staff_Name",
                "ChangeEffectiveFrom", "ChangeEffectiveTo", "HashDiff"
            )
            SELECT 
                stg.id, stg.name,
                DATETIME('now'), '9999-12-31 23:59:59',
                sha256(COALESCE(stg.name,''))
            FROM "stg_staff" stg
            LEFT JOIN "silver_staff" ss ON stg.id = ss.Staff_ID AND ss.ChangeEffectiveTo = '9999-12-31 23:59:59'
            WHERE ss.Staff_ID IS NULL
               OR ss.HashDiff <> sha256(COALESCE(stg.name,''))"""
        ),
    ]

    total_updated = 0
    total_inserted = 0
    failed_tables = []

    for table_name, update_sql, insert_sql in tables:
        updated, inserted, success = execute_table_load(conn, update_sql, insert_sql, table_name)
        if not success:
            failed_tables.append(table_name)
        total_updated += updated
        total_inserted += inserted

    conn.close()

    summary = f"Pipeline complete | Total rows closed off: {total_updated} | Total rows inserted: {total_inserted}"
    print(summary)
    logger.info(summary)

    if failed_tables:
        warning = f"Tables that errored: {failed_tables}"
        print(warning)
        logger.warning(warning)
    else:
        success_msg = "All tables loaded successfully"
        print(success_msg)
        logger.info(success_msg)


run_silver_loads(r"C:\Users\liam_\Documents\Games DE Project\Zelda DB\ZeldaDatabase.db")

2026-02-21 18:56:12,809 - INFO - Connected to: C:\Users\liam_\Documents\Games DE Project\Zelda DB\ZeldaDatabase.db
2026-02-21 18:56:12,815 - INFO - silver_games: 12 rows already in DB | no changes detected, rows already up to date
2026-02-21 18:56:12,825 - INFO - silver_character: 1657 rows already in DB | no changes detected, rows already up to date
2026-02-21 18:56:12,828 - INFO - silver_dungeons: 335 rows already in DB | no changes detected, rows already up to date
2026-02-21 18:56:12,831 - INFO - silver_Bosses: 255 rows already in DB | no changes detected, rows already up to date
2026-02-21 18:56:12,836 - INFO - silver_monsters: 815 rows already in DB | no changes detected, rows already up to date
2026-02-21 18:56:12,847 - INFO - silver_items: 1849 rows already in DB | no changes detected, rows already up to date
2026-02-21 18:56:12,859 - INFO - silver_places: 1470 rows already in DB | no changes detected, rows already up to date
2026-02-21 18:56:12,862 - INFO - silver_staff: 234 r

Connected to: C:\Users\liam_\Documents\Games DE Project\Zelda DB\ZeldaDatabase.db
silver_games: 12 rows already in DB | no changes detected, rows already up to date
silver_character: 1657 rows already in DB | no changes detected, rows already up to date
silver_dungeons: 335 rows already in DB | no changes detected, rows already up to date
silver_Bosses: 255 rows already in DB | no changes detected, rows already up to date
silver_monsters: 815 rows already in DB | no changes detected, rows already up to date
silver_items: 1849 rows already in DB | no changes detected, rows already up to date
silver_places: 1470 rows already in DB | no changes detected, rows already up to date
silver_staff: 234 rows already in DB | no changes detected, rows already up to date
Pipeline complete | Total rows closed off: 0 | Total rows inserted: 0
All tables loaded successfully
